# Organoid Regulome Benchmark: hgx on Fleck et al. 2023
GPU-accelerated hypergraph deep learning on cerebral organoid gene regulatory networks.
Requires Colab Pro/Pro+ with A100 runtime.

In [ ]:
# Mount Google Drive for persistent data storage
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_DIR = "/content/drive/MyDrive/organoid-hgx-benchmark"
os.makedirs(DRIVE_DIR, exist_ok=True)
os.makedirs(f"{DRIVE_DIR}/data", exist_ok=True)
os.makedirs(f"{DRIVE_DIR}/figures", exist_ok=True)

# Install dependencies
!pip install -q jax[cuda12] equinox diffrax optax jaxtyping scanpy anndata ripser matplotlib

# Clone and install hgx
if not os.path.exists("/content/hgx"):
    !git clone https://github.com/m9h/hgx.git /content/hgx
!pip install -q -e /content/hgx

# Clone benchmark repo
if not os.path.exists("/content/organoid-hgx-benchmark"):
    !git clone https://github.com/m9h/organoid-hgx-benchmark.git /content/organoid-hgx-benchmark

# Verify GPU
import jax
print(f"JAX devices: {jax.devices()}")
print(f"Backend: {jax.default_backend()}")

In [ ]:
import subprocess

DATA_DIR = f"{DRIVE_DIR}/data"

files = {
    "pando/coefs.tsv": "https://zenodo.org/records/15371701/files/grn_modules.tsv?download=1",
    "expression/RNA_data.h5ad": "https://zenodo.org/records/15371701/files/RNA_data.h5ad?download=1",
    "zenodo/rna_matrices.tar.gz": "https://zenodo.org/records/15371701/files/rna_matrices.tar.gz?download=1",
    "zenodo/RNA_all_velo.h5ad": "https://zenodo.org/records/15371701/files/RNA_all_velo.h5ad?download=1",
    "zenodo/motifs.tar.gz": "https://zenodo.org/records/15371701/files/motifs.tar.gz?download=1",
}

for path, url in files.items():
    full_path = f"{DATA_DIR}/{path}"
    os.makedirs(os.path.dirname(full_path), exist_ok=True)
    if os.path.exists(full_path):
        size_mb = os.path.getsize(full_path) / 1e6
        print(f"  EXISTS: {path} ({size_mb:.0f} MB)")
    else:
        print(f"  Downloading: {path}...")
        subprocess.run(["curl", "-L", "-o", full_path, url], check=True)
        size_mb = os.path.getsize(full_path) / 1e6
        print(f"    Done: {size_mb:.0f} MB")

# Extract RNA matrices if not already done
meta_dir = f"{DATA_DIR}/zenodo/data_matrices"
if not os.path.exists(meta_dir):
    print("Extracting rna_matrices.tar.gz...")
    subprocess.run(["tar", "xzf", f"{DATA_DIR}/zenodo/rna_matrices.tar.gz",
                     "-C", f"{DATA_DIR}/zenodo/"], check=True)
    print(f"  Extracted to {meta_dir}")
else:
    print(f"  data_matrices/ already exists")

print("\nAll data ready!")

In [ ]:
# Symlink data so preprocessing script finds it
import shutil
work_dir = "/content/organoid-hgx-benchmark"
data_link = f"{work_dir}/data"
if os.path.islink(data_link):
    os.unlink(data_link)
elif os.path.isdir(data_link):
    shutil.rmtree(data_link)
os.symlink(DATA_DIR, data_link)

# Run preprocessing
!cd {work_dir} && python scripts/00_preprocess.py --num-bins 10 --feature-dim 16

In [ ]:
# Symlink figures dir to Drive for persistence
fig_link = f"{work_dir}/figures"
if os.path.islink(fig_link):
    os.unlink(fig_link)
elif os.path.isdir(fig_link):
    shutil.rmtree(fig_link)
os.symlink(f"{DRIVE_DIR}/figures", fig_link)

# Run the full benchmark
!cd {work_dir} && python scripts/run_all_real.py --epochs 200 --seed 42

## Results
All figures saved to Google Drive at `organoid-hgx-benchmark/figures/`.

In [ ]:
from IPython.display import display, Image
from pathlib import Path

fig_dir = Path(f"{DRIVE_DIR}/figures")
for fig_path in sorted(fig_dir.glob("*.png")):
    print(f"\n{'='*60}")
    print(f"  {fig_path.name}")
    print(f"{'='*60}")
    display(Image(filename=str(fig_path), width=900))

## Quantitative Summary

| Analysis | Metric | Result |
|----------|--------|--------|
| Module Detection | Macro F1 | See output above |
| Attention Correlation | Pearson r | See output above |
| Neural ODE | Rollout MSE | See output above |
| GLI3 KO | Pearson r | See output above |
| Perturbation Screen | 720 TFs screened | See output above |
| Topology | β₀, β₁ from Hodge | See output above |
| NDP | Cell growth 5→500 | See output above |

### Data Sources
- **Pando GRN:** 74,448 edges, 720 TFs, 2,792 genes (Zenodo)
- **Expression:** 49,718 cells × 33,538 genes (Zenodo)
- **Metadata:** 34,088 cells with pseudotime, lineage, fate probabilities
- **Paper:** Fleck et al., Nature 2023 (doi:10.1038/s41586-022-05279-8)